# 文档加载器 Document Loaders

## 1.记载txt



In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(
    file_path = "../asset/load/01-langchain-utf-8.txt",
    encoding = "utf-8"
)

docs = loader.load()

print(docs)

Document对象有两个重要的属性：
1）page_content: 真正的文档内容，字符串类型。
2）metadata: 文档内容的元数据，字典类型

In [ ]:
print(docs[0])
print(type(docs[0]))

print(docs[0].metadata)
print(docs[0].page_content)

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(
    file_path = "../asset/load/01-langchain-gbk.txt",
    encoding = "gbk",   # 务必编码和解码一致
)

docs = loader.load()

print(docs)

## 2.加载CSV格式

In [ ]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(
    file_path = "../asset/load/02-load.csv"
)

docs = loader.load()

print(docs)

## 3.加载JSON

LangChain提供的JSON格式的文档加载器是`JSONLoader`。在实际应用场景中，JSON格式的数据占有很大比例，而且JSON的形式也是多样的。我们需要特别关注。

JSONLoader 使用指定的 `jq结构`来解析 JSON 文件。jq是一个轻量级的命令行 JSON 处理器 ，可以对 JSON 格式的数据进行各种复杂的处理，包括数据过滤、映射、减少和转换，是处理 JSON 数据的`首选工具之一`。

常见 jq schema 参考：

JSON        -> [{"text": ...}, {"text": ...}, {"text": ...}]
jq_schema   -> ".[].text"

JSON        -> {"key": [{"text": ...}, {"text": ...}, {"text": ...}]}
jq_schema   -> ".key[].text"

JSON        -> ["...", "...", "..."]
jq_schema   -> ".[]"


举例1：使用JSONLoader文档器加载

In [ ]:
from langchain_community.document_loaders import JSONLoader
from rich import print as rprint

# 情况1
# json_loader = JSONLoader(
#     file_path = "../asset/load/03-load.json",
#     jq_schema = ".",   # 直接提取完整的JSON对象（包括所有字段）
#     text_content=False    # 保持原始JSON结构，将提取的数据转换成JSON字符串存入page_content字段中
# )

# 情况2
# .messages[].content：遍历.messages[]中所有元素，从每一个元素中提取.content字段
json_loader = JSONLoader(
    file_path = "../asset/load/03-load.json",
    jq_schema=".messages[].content"
)


# 加载
docs = json_loader.load()
rprint(docs)

举例2：提取03-response.json文件中指定的文本

In [ ]:
from langchain_community.document_loaders import JSONLoader
from rich import print as rprint

file_path = "../asset/load/03-response.json"

# 需求1：提取data.items
# loader = JSONLoader(
#     file_path = file_path,
#     jq_schema = ".data.items[]",
#     text_content = False   # 提取内容是否为字符串格式，默认是True
# )

# 需求2：提取data.items[].content中的数据
# loader = JSONLoader(
#     file_path = file_path,
#     jq_schema = ".data.items[].content",
# )

# 需求3：提取data.items中指定字段的数据
loader = JSONLoader(
    file_path = file_path,
    jq_schema = """
        .data.items[] | {
            author,
            created_at,
            content: (.title + "\n" + .content)
        }
    """,
    text_content = False,    # 提取内容是否为字符串格式
)

data = loader.load()
rprint(data)

## 4、加载pdf

PDF 存在多种来源格式，包括扫描版（图片 PDF）、电子文本版、混合版。并且布局格式也多种多样，包括单列布局、双列布局甚至竖排文本布局。并且包含段落、标题、页眉页脚、表格、数学公式、化学式、特殊符号、图片等各种元素。

因此，PDF 解析存在很多挑战。对于复杂 PDF，需要进行文本提取、布局检测、表格解析、公式识别等处理。

方式一：使用PyPDFLoader

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    # 文件路径
    # file_path = "../asset/load/04-sample.pdf",
    file_path = "https://arxiv.org/pdf/alg-geom/9202012",
    # 提取模式
    #  plain 提取文本，默认值
    #  layout 布局感知提取模式，通常会通过插入大量的空格、换行符，来模拟原本档中的多栏、缩进和间距（使用场景：学术论文、多栏报刊杂志、带有左右分栏的合同）
    extraction_mode = "plain",
)

docs = loader.load()

print(docs)
print(len(docs))

方式2：使用MinerU

[MinerU](https://mineru.net/) 提供了 PDF、Word、PPT、图片等文件的解析，支持图像提取、OCR、公式、表格解析等功能。

调用在线服务：https://mineru.net/apiManage/docs。可以从本地批量上传文件进行解析，并接收解析结果。

In [ ]:
import os
import time
import requests
from dotenv import load_dotenv

load_dotenv(override=True)


def upload_files(file_paths: list[str]) -> str:
    """批量上传文件"""
    url = "https://mineru.net/api/v4/file-urls/batch"
    api_token = os.getenv("MINERU_API_TOKEN")
    header = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_token}",
    }

    files_info = [
        {
            "name": os.path.basename(file_path),
            "is_ocr": True,
            "data_id": f"file_{i}",
        }
        for i, file_path in enumerate(file_paths)
    ]

    data = {
        "enable_formula": True,
        "enable_table": True,
        "language": "ch",
        "files": files_info,
    }

    try:
        response = requests.post(url, headers=header, json=data)
        if response.status_code == 200:
            result = response.json()
            print("response success. result:{}".format(result))

            if result["code"] == 0:
                batch_id = result["data"]["batch_id"]
                urls = result["data"]["file_urls"]
                print("batch_id:{}\nurls:{}".format(batch_id, urls))

                for i in range(0, len(urls)):
                    with open(file_paths[i], "rb") as f:
                        res_upload = requests.put(urls[i], data=f)
                        if res_upload.status_code == 200:
                            print(f"{urls[i]} upload success")
                        else:
                            print(f"{urls[i]} upload failed")
                            return None

                return batch_id
            else:
                print("apply upload url failed, reason:{}".format(result.get("msg")))
                return None
        else:
            print(
                "response not success. status:{} ,result:{}".format(
                    response.status_code, response.text
                )
            )
            return None

    except Exception as err:
        print(err)
        return None


def download_files(batch_id):
    """批量获取任务结果"""
    if not batch_id:
        print("batch_id为空，跳过下载")
        return

    os.makedirs("parsed_files", exist_ok=True)

    url = f"https://mineru.net/api/v4/extract-results/batch/{batch_id}"
    api_token = os.getenv("MINERU_API_TOKEN")
    header = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_token}",
    }

    failed_files = set()
    done_files = set()

    while True:
        res = requests.get(url, headers=header)
        result_json = res.json()

        if res.status_code != 200 or result_json.get("code") != 0:
            print("get result failed:", result_json)
            break

        extract_results = result_json["data"]["extract_result"]

        for result in extract_results:
            data_id = result["data_id"]

            if result["state"] == "failed":
                failed_files.add(data_id)

            elif result["state"] == "done" and data_id not in done_files:
                done_files.add(data_id)

                full_zip_url = result["full_zip_url"]
                res_download = requests.get(full_zip_url, stream=True)

                with open(
                    f"parsed_files/{result['file_name']}_{result['data_id']}.zip", "wb"
                ) as f:
                    for chunk in res_download.iter_content(chunk_size=1024):
                        if chunk:
                            f.write(chunk)

        if len(failed_files) + len(done_files) == len(extract_results):
            break

        time.sleep(5)

    for i in failed_files:
        print("failed:", i)

    for i in done_files:
        print("done:", i)


file_paths = ["../asset/load/04-sample.pdf"]
batch_id = upload_files(file_paths)

if batch_id:
    download_files(batch_id)